In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

from tdmclient import ClientAsync
import asyncio, threading, time

# Variables


--- 

x : state (pose : [x, y, $\theta$])

P : 3x3 covariance matrix for uncertainity on state 

Q : 3x3 covariance matrix for process noise (usually diagonal)

R : 3x3 covariance matrix for noise on camera (usually diagonal)

wheel_base : distance between the wheels (~95mm)

---

v_l : left speed of motors converted to [mm/s]

v_r : right speed of motors converted to [mm/s]

v : forward speed of the robot [mm/s]

w : rotation rate of the robot [rad/s]

J : Jacobian of the motion model

---

z : camera measurements ([x, y, $\theta$])

H : measurement matrix (maps the state to the observations)

y : innovation (diff between state and info from observation)

S : innovation covariance (expected uncertainty -> can be used as outlier gate)

K : kalman gain 

---


<br>
<br>
<br>
<br>
<br>

# Concept 

--- 


predict : uses the wheel speed to move the state (increases state uncertainty P using process uncertainty Q). Since our motion model isn't linear we use the EKF to resolve it. the model is :

<br>

$$x_{k+1} = x_{k} + v_{k} \times \cos(\theta_{k}) \times \delta t$$
$$y_{k+1} = y_{k} + v_{k} \times \sin(\theta_{k})\times \delta t$$
$$\theta_{k+1} = \theta_{k} + \omega_{k} \times \delta t $$

<br>

Since our model isn't linear we need to locally linearize it to propagate uncertainty in the state position :   $$P_{k+1} = J \ P_{k} \ J^T + Q$$

We use the Jacobian J which is the best linear approximation around a given state

$$ J = \begin{bmatrix} \frac{\delta f_i}{\delta x_j} \end{bmatrix}_{i,j} 
\implies J = \begin{bmatrix} 
                1 & 0 &  -v \times \sin(\theta) \times \delta t \\
                0 & 1 & v \times \cos(\theta) \times \delta t \\
                0 & 0 & 1
             \end{bmatrix}$$

<br>

--- 


update : uses the camera measurements (z) to "correct" the predictions done. It defines the innovation as the difference between the observation (z) and the prediction (x) using the relation between the two (H) :

$$ y = z - H \ x $$

Using the innovation covariance matrix (S) computed using the observation uncertainty (R) it computes the kalman gain (K) :

$$ S = H \ P \ H^T + R $$

$$ K = P \ H^T \ S^{-1} $$

The kalman gain (K) is then used to update the state (x) and its uncertainty (P) using previously computed quantities : 

$$ x \leftarrow x + K \ y $$

$$ P \leftarrow (I - K \ H) \ P $$

<br>

---

In [ ]:
# class of the Extended Kalman Filter

def wrap(a):
    return (a + np.pi) % (2*np.pi) - np.pi

class EKF:
    

    def __init__(self, x0, P0, Q, R, wheel_base, H):
        
        self.x = np.array(x0, dtype=float) 
        self.P = np.array(P0, dtype=float)
        self.Q, self.R, self.L, self.H = Q, R, wheel_base, H
        

    def predict(self, v_l, v_r, dt):
        
        v = (v_r + v_l) / 2
        w = (v_r - v_l) / self.L
        x, y, theta = self.x
        
        self.x = np.array([x + v*np.cos(theta)*dt,y + v*np.sin(theta)*dt, wrap(theta + w*dt)])
        
        J = np.array([[1, 0, -v*np.sin(theta)*dt], [0, 1,  v*np.cos(theta)*dt], [0, 0,  1]])
        self.P = J @ self.P @ J.T + self.Q
        

    def update(self, z):      
                        
        y = z - self.H @ self.x
        y[2] = wrap(y[2])
        S = self.H @ self.P @ self.H.T + self.R
        S_inv = np.linalg.inv(S)
        
        if y @ S_inv @ y > GATE:
            return False
        
        K = self.P @ self.H.T @ S_inv
        
        self.x = self.x + K @ y
        self.x[2] = wrap(self.x[2])
        self.P = (np.eye(3) - K @ self.H) @ self.P
        return True

In [ ]:
# thymio data

SPEED_TO_MM_S = 0.35    # [Thymio raw unit -> mm/s] -> tune it on our robot (~0.3-0.4)
WHEEL_BASE = 95.0   # [mm] -> distance between wheels -> tune it on our robot
LOOP_DT = 0.05  # [s] -> target loop period for the filter
GATE = 11.34    # threshold for innovation -> chi-square (3 DoF) for 99% 


In [ ]:
# thymio/camera functions 

def read_wheels_speed():
    vl = node.v.motor.left.speed * SPEED_TO_MM_S
    vr = node.v.motor.right.speed * SPEED_TO_MM_S
    return vl, vr

async def set_motors_speed(left, right):
    node.v.motor.left.target = int(left)
    node.v.motor.right.target = int(right)
    await node.flush()
    


def get_camera_pose(): # to change when we know what it looks like
    pass


<br>
<br>
<br>
<br>
<br>
<br>

In [ ]:
# connect the thymio

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()
await node.wait_for_variables({"motor.left.speed", "motor.right.speed"})
print("connected")

In [ ]:
# camera thread

latest = {"z": None, "stamp": 0, "t": 0.0} 
cam_running = True

def camera_worker():
    while cam_running:
        z = get_camera_pose()
        if z is not None:
            latest["z"] = z
            latest["t"] = time.perf_counter()
            latest["stamp"] += 1 

cam_thread = threading.Thread(target=camera_worker, daemon=True)
cam_thread.start()

In [ ]:
# initialize filter 

while latest["z"] is None:
    await asyncio.sleep(0.05)

ekf = EKF(x0=latest["z"],
          P0=np.diag([25, 25, 0.01]),
          Q=np.zeros((3, 3)),
          R=np.diag([9, 9, 0.005]),
          wheel_base=WHEEL_BASE,
          H=np.eye(3))

q_rate = np.diag([20, 20, 0.01])    # process noise per second

In [ ]:
# main loop

running = True
cmd = {"left": 0, "right": 0}   # [thymio units] -> navigation writes here -> to see with the others
log = {"t": [], "x": [], "z": [], "Pdiag": [], "rejected": []}
last_stamp = 0
MAX_AGE = 0.3   # [s] -> ignore camera frames older than this 

async def main_loop():
    global last_stamp
    t0 = t_prev = time.perf_counter()
    try:
        while running:
            await client.sleep(LOOP_DT)
            t = time.perf_counter()
            dt, t_prev = t - t_prev, t

            # predict from wheel odometry
            v_l, v_r = read_wheels_speed()
            ekf.Q = q_rate * dt
            ekf.predict(v_l, v_r, dt)

            # update with new/recent camera observation
            z, rejected = None, False
            if latest["stamp"] != last_stamp and t - latest["t"] < MAX_AGE:
                last_stamp = latest["stamp"]
                z = latest["z"].copy()
                rejected = not ekf.update(z)

            # send motor commands from navigation -> to see with the others
            await set_motors_speed(cmd["left"], cmd["right"])

            # log from the filters
            log["t"].append(t - t0)
            log["x"].append(ekf.x.copy())
            log["z"].append(z if z is not None else np.full(3, np.nan))
            log["Pdiag"].append(np.diag(ekf.P).copy())
            log["rejected"].append(rejected)
            
    except Exception as e:
        print("Main loop crashed : ", repr(e))
        raise
    finally:
        await set_motors_speed(0, 0)


task = asyncio.create_task(main_loop())

In [ ]:
# stop

running = False
await task              
cam_running = False
await node.unlock()

# to do : 

- check if its okay to use camera for filtering

- define camera functions
    - camera frame convertions (more from vision's task)
    
- see "to see with others" things